In [5]:
#2 Data sanity check
import numpy as np
import os

# Data Sanity Checks
def load_split(split_path):
    signals = [
        "body_acc_x", "body_acc_y", "body_acc_z",
        "body_gyro_x", "body_gyro_y", "body_gyro_z",
        "total_acc_x", "total_acc_y", "total_acc_z"
    ]

    data_list = []

    for sig in signals:
        file_path = os.path.join(f"{sig}_{os.path.basename(split_path)}.txt")
        data = np.loadtxt(file_path)
        data_list.append(data)

    # Stack as (num_windows, T, C)
    X = np.stack(data_list, axis=-1)
    return X
X_train = load_split("train")
X_test = load_split("test")
y_train = np.loadtxt("y_train.txt")
y_test = np.loadtxt("y_test.txt")

In [6]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------------
# Model
# --------------------------------------------------
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes, dropout=0.3):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: (batch, T, C)
        lstm_out, _ = self.lstm(x)

        # use the last timestep output
        last_hidden = lstm_out[:, -1, :]   # shape: (batch, hidden_dim)

        out = self.dropout(last_hidden)
        logits = self.fc(out)
        return logits


model = LSTMClassifier(
    input_dim=9,
    hidden_dim=128,
    num_layers=2,
    num_classes=6,
    dropout=0.3
).to(device)

# --------------------------------------------------
# Example: move tensors to device
# --------------------------------------------------
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.long).to(device)
X_val   = torch.tensor(X_test, dtype=torch.float32).to(device)
y_val   = torch.tensor(y_test, dtype=torch.long).to(device)
X_test  = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test  = torch.tensor(y_test, dtype=torch.long).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

# Reduce LR when validation loss stops improving
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

# --------------------------------------------------
# Training settings
# --------------------------------------------------
epochs = 50
max_grad_norm = 1.0      # gradient clipping threshold
early_stop_patience = 7

best_val_loss = float("inf")
best_model_wts = copy.deepcopy(model.state_dict())
epochs_no_improve = 0

for epoch in range(epochs):
    # -------------------------
    # Train
    # -------------------------
    model.train()
    optimizer.zero_grad()

    train_logits = model(X_train)
    train_loss = criterion(train_logits, y_train)

    train_loss.backward()

    # Gradient clipping for RNN/LSTM robustness
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

    optimizer.step()

    train_preds = torch.argmax(train_logits, dim=1)
    train_acc = (train_preds == y_train).float().mean().item()

    # -------------------------
    # Validate
    # -------------------------
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = criterion(val_logits, y_val)

        val_preds = torch.argmax(val_logits, dim=1)
        val_acc = (val_preds == y_val).float().mean().item()

    # Step scheduler on validation loss
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss.item():.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss.item():.4f} | Val Acc: {val_acc:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    # -------------------------
    # Early stopping
    # -------------------------
    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        best_model_wts = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= early_stop_patience:
        print(f"Early stopping triggered at epoch {epoch+1}.")
        break

# Load best model
model.load_state_dict(best_model_wts)

# --------------------------------------------------
# Test
# --------------------------------------------------
model.eval()
with torch.no_grad():
    test_logits = model(X_test)
    test_preds = torch.argmax(test_logits, dim=1)
    test_acc = (test_preds == y_test).float().mean().item()

print("Best validation loss:", best_val_loss)
print("Test accuracy:", test_acc)

IndexError: Target 6 is out of bounds.